# Tutorial 2 — Converting Between Annotation Formats

**Scenario:** You found a YOLO-labelled dataset on Roboflow Universe
and want to enter a computer vision competition that requires COCO JSON
submissions.  Your training framework also wants HuggingFace Parquet.

Rather than writing one-off conversion scripts (and debugging them at
midnight before the deadline), you'll use `dman` as a single-command
conversion bridge:

```
YOLO on disk  →  dman import  →  dman export --format coco
                                 dman export --format huggingface
```

This tutorial shows how that pipeline works, explains the quirks of
each format, and points out what to watch for when the conversion isn't
lossless.

**Prerequisites:** `pip install dman pillow`

## Setup

In [ ]:
import os
import tempfile
import pathlib
import subprocess

tutorial_dir = pathlib.Path(tempfile.mkdtemp(prefix="dman_tutorial2_"))
os.environ["DMAN_HOME"] = str(tutorial_dir / "catalog")

subprocess.run(["dman", "init"], check=True)
print("Catalog initialised at", os.environ["DMAN_HOME"])

## Step 1 — Create a realistic YOLO dataset on disk

YOLO format stores:
- images in `images/train/`
- one `.txt` label file per image in `labels/train/`
- a `data.yaml` with class names and split paths

Each label line: `<class_id> <cx> <cy> <w> <h>` (all normalised 0–1).

In [ ]:
import random
import yaml
from PIL import Image, ImageDraw

random.seed(0)

yolo_root = tutorial_dir / "pothole_yolo"
(yolo_root / "images" / "train").mkdir(parents=True)
(yolo_root / "labels" / "train").mkdir(parents=True)

CLASSES    = ["pothole", "crack", "patch"]
IMG_W, IMG_H = 256, 192

# 10 synthetic road images with random defects
SAMPLES = []
for i in range(10):
    stem = f"road_{i:03d}"

    # Image: grey road surface with noise
    img = Image.new("RGB", (IMG_W, IMG_H),
                    (random.randint(80, 130),) * 3)
    draw = ImageDraw.Draw(img)

    annotations = []
    n_defects = random.randint(1, 3)
    for _ in range(n_defects):
        cls_id = random.randint(0, len(CLASSES) - 1)
        cx = random.uniform(0.1, 0.9)
        cy = random.uniform(0.1, 0.9)
        bw = random.uniform(0.05, 0.25)
        bh = random.uniform(0.05, 0.20)

        # Draw defect
        x1 = int((cx - bw / 2) * IMG_W)
        y1 = int((cy - bh / 2) * IMG_H)
        x2 = int((cx + bw / 2) * IMG_W)
        y2 = int((cy + bh / 2) * IMG_H)
        color = [(40, 20, 20), (20, 40, 20), (20, 20, 40)][cls_id]
        draw.rectangle([x1, y1, x2, y2], fill=color)

        annotations.append((cls_id, cx, cy, bw, bh))

    img.save(yolo_root / "images" / "train" / f"{stem}.jpg")

    label_lines = [f"{c} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"
                   for c, cx, cy, bw, bh in annotations]
    (yolo_root / "labels" / "train" / f"{stem}.txt").write_text(
        "\n".join(label_lines)
    )
    SAMPLES.append((stem, annotations))

# data.yaml — the file dman reads to understand the dataset
(yolo_root / "data.yaml").write_text(
    yaml.dump({
        "path":  str(yolo_root),
        "train": "images/train",
        "nc":    len(CLASSES),
        "names": CLASSES,
    })
)

total_annots = sum(len(a) for _, a in SAMPLES)
print(f"Created {len(SAMPLES)} images with {total_annots} annotations")
print("Classes:", CLASSES)

## Step 2 — Import into dman

`dman import` parses the on-disk format, normalises every annotation
into dman's internal representation, and registers the dataset in the catalog.

From this point on, the *original YOLO files are not modified*.
dman stores references to them, not copies.

In [ ]:
result = subprocess.run(
    ["dman", "import", str(yolo_root), "--format", "yolo", "--name", "potholes"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

In [ ]:
# Confirm it's registered
result = subprocess.run(["dman", "inspect", "potholes"], capture_output=True, text=True)
print(result.stdout)

## Step 3 — Verify the import with the Python SDK

Before converting, it's worth confirming the annotation counts match
what we created.  A mismatch here would mean the importer dropped
something — better to catch it now than after submission.

In [ ]:
import dman
import json

ds = dman.load_dataset("potholes")

print(f"Samples     : {ds.sample_count()}")
print(f"Annotations : {ds.annotation_count()}")
print(f"Categories  : {ds.category_count()}")

# Spot-check: compare annotation count per sample against our source data
print("\nPer-sample annotation counts (dman vs. source):")
print(f"  {'Sample':<12} {'dman':>5}  {'source':>6}  {'match?':>6}")
print("  " + "-" * 38)
all_match = True
for stem, source_annots in SAMPLES:
    dman_annots = ds.annotations(stem)
    match = len(dman_annots) == len(source_annots)
    all_match = all_match and match
    flag = "✓" if match else "✗ MISMATCH"
    print(f"  {stem:<12} {len(dman_annots):>5}  {len(source_annots):>6}  {flag}")

print("\nAll counts match:", all_match)

## Step 4 — Export to COCO JSON

COCO stores everything in a single `annotations.json` with four
top-level keys: `info`, `categories`, `images`, `annotations`.

Notice the coordinate change: YOLO uses normalised centre-x/y,
while COCO uses absolute top-left x/y.  `dman export` handles
this conversion automatically.

In [ ]:
coco_out = tutorial_dir / "potholes_coco"

result = subprocess.run(
    ["dman", "export", "potholes", str(coco_out), "--format", "coco"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

In [ ]:
# Inspect the COCO output
coco_json_path = next(coco_out.rglob("*.json"))
coco = json.loads(coco_json_path.read_text())

print("COCO JSON top-level keys:", list(coco.keys()))
print(f"  images      : {len(coco['images'])}")
print(f"  annotations : {len(coco['annotations'])}")
print(f"  categories  : {len(coco['categories'])}")
print()

# Show the first annotation to illustrate the format
print("Example annotation (COCO format):")
print(json.dumps(coco["annotations"][0], indent=2))

### What changed between YOLO and COCO?

| Field | YOLO | COCO |
|-------|------|------|
| Bbox  | `[cx, cy, w, h]` normalised (0–1) | `[x, y, w, h]` in pixels, top-left origin |
| Class | integer index into `names` list | `category_id` with a separate `categories` list |
| Structure | one `.txt` per image | one `annotations.json` for all images |

dman handles the de-normalisation using the image dimensions it recorded
at import time — which is why those dimensions matter.

## Step 5 — Export to HuggingFace Parquet

If you want to use `datasets.load_from_disk()` for training, export
to HuggingFace format.  It produces Parquet files that work directly
with the `datasets` library.

In [ ]:
hf_out = tutorial_dir / "potholes_hf"

result = subprocess.run(
    ["dman", "export", "potholes", str(hf_out), "--format", "huggingface"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

# Show what was produced
print("\nOutput files:")
for p in sorted(hf_out.rglob("*")):
    if p.is_file():
        size = p.stat().st_size
        print(f"  {p.relative_to(hf_out)}  ({size} bytes)")

## Step 6 — Round-trip check

A common concern: *did the conversion lose any annotations?*

We can verify by importing the COCO output back into a fresh dataset
and comparing annotation counts.

In [ ]:
# Import the COCO export back in
result = subprocess.run(
    ["dman", "import", str(coco_out), "--format", "coco", "--name", "potholes_roundtrip"],
    capture_output=True, text=True
)
print(result.stdout)

rt = dman.load_dataset("potholes_roundtrip")

print(f"\nOriginal  — samples: {ds.sample_count()}, annotations: {ds.annotation_count()}")
print(f"Round-trip — samples: {rt.sample_count()}, annotations: {rt.annotation_count()}")
print("Counts match:", ds.annotation_count() == rt.annotation_count())

## Cleanup

In [ ]:
import shutil
shutil.rmtree(tutorial_dir)
print("Cleaned up:", tutorial_dir)

---

**What you learned:**

- `dman import --format yolo` parses a YOLO directory into the catalog
- `dman export --format coco` converts to COCO JSON (bbox coordinates are de-normalised automatically)
- `dman export --format huggingface` produces Parquet files for the `datasets` library
- The Python SDK lets you verify annotation counts before trusting the output

**Next:** [Tutorial 3 — Analysing Dataset Quality](tutorial_03_dataset_analysis.ipynb)